# 🛒 Supermarket Sales Analysis

**Dataset:** `data/supermarket_data.csv` — 500 transactions · 4 cities · 8 categories · Jan–Jul 2026  
**Tech Stack:** Python · Pandas · Matplotlib · Seaborn · Plotly

---

## Analysis Pipeline

| Step | Description |
|------|-------------|
| 1 | Collect and load the CSV dataset |
| 2 | Check the data for missing or incorrect values |
| 3 | Calculate Sales = Quantity × Unit Price |
| 4 | Group and summarize (totals, counts, averages) |
| 5 | Create charts to compare results |
| 6 | Use results to make business decisions |

---
## 📦 0. Install & Import Libraries

In [ ]:
# Install required libraries (uncomment if not already installed)
# !pip install pandas matplotlib seaborn plotly

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Style ──────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
})

BLUE   = '#1E3A5F'
ACCENT = '#3B82F6'
COLORS = ['#1E3A5F','#2563EB','#3B82F6','#60A5FA','#93C5FD','#BFDBFE','#DBEAFE','#EFF6FF']

print('✅ Libraries imported successfully')

---
## 📂 Step 1 — Collect and Load the Dataset

In [ ]:
# ── Load CSV ───────────────────────────────────────
DATA_PATH = os.path.join('data', 'supermarket_data.csv')

df_raw = pd.read_csv(DATA_PATH, parse_dates=['Date'])

print(f'Shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Date range : {df_raw["Date"].min().date()}  →  {df_raw["Date"].max().date()}')
print()
print('Column names:')
print(df_raw.columns.tolist())

In [ ]:
# First 5 rows
df_raw.head()

In [ ]:
# Data types & memory
df_raw.info()

In [ ]:
# Summary statistics for numeric columns
df_raw.describe().round(2)

---
## 🔍 Step 2 — Data Quality Check

In [ ]:
# ── 2.1 Missing Values ─────────────────────────────
missing = df_raw.isnull().sum()
print('=== Missing Values per Column ===')
print(missing)
print(f'\nTotal missing cells : {missing.sum()}')

In [ ]:
# ── 2.2 Duplicate Invoice IDs ──────────────────────
dupes = df_raw.duplicated(subset=['Invoice ID']).sum()
print(f'Duplicate Invoice IDs : {dupes}')

In [ ]:
# ── 2.3 Negative Numeric Fields ───────────────────
print(f"Negative Quantity rows   : {(df_raw['Quantity']   < 0).sum()}")
print(f"Negative Unit Price rows : {(df_raw['Unit Price'] < 0).sum()}")
print(f"Negative Sales rows      : {(df_raw['Sales']      < 0).sum()}")

In [ ]:
# ── 2.4 Categorical Value Counts ──────────────────
for col in ['Branch', 'City', 'Customer Type', 'Gender', 'Category', 'Payment']:
    print(f'--- {col} ---')
    print(df_raw[col].value_counts().to_string())
    print()

In [ ]:
# ── 2.5 Sales Mismatch Check ──────────────────────
df_raw['_check'] = (df_raw['Quantity'] * df_raw['Unit Price']).round(2)
mismatch = (df_raw['Sales'].round(2) != df_raw['_check']).sum()
print(f'Rows where stored Sales ≠ Quantity × Unit Price : {mismatch}')
df_raw.drop(columns=['_check'], inplace=True)

print()
print('✅ Data quality check complete. Dataset is clean.')

---
## 🧮 Step 3 — Calculate Sales = Quantity × Unit Price

In [ ]:
# ── Clean & enrich the DataFrame ──────────────────
df = df_raw.copy()

# Drop duplicates
df = df.drop_duplicates()

# Keep only valid rows
df = df[(df['Quantity'] > 0) & (df['Unit Price'] > 0)]

# Recalculate Sales authoritatively
df['Sales'] = (df['Quantity'] * df['Unit Price']).round(2)

# Derived time columns
df['Month']       = df['Date'].dt.month
df['Month_Label'] = df['Date'].dt.to_period('M').astype(str)
df['Quarter']     = df['Date'].dt.to_period('Q').astype(str)
df['Day_of_Week'] = df['Date'].dt.day_name()

df = df.reset_index(drop=True)

print(f'Clean dataset : {len(df):,} rows')
print()
# Show Sales calculation sample
print('Sales = Quantity × Unit Price  (first 5 rows):')
df[['Invoice ID', 'Product', 'Quantity', 'Unit Price', 'Sales']].head()

---
## 📊 Step 4 — Group & Summarise (Totals, Counts, Averages)

In [ ]:
# ── 4.1 Overall KPIs ──────────────────────────────
kpis = {
    'Total Revenue (₹)'     : df['Sales'].sum().round(2),
    'Total Transactions'    : len(df),
    'Avg Order Value (₹)'   : df['Sales'].mean().round(2),
    'Total Units Sold'      : int(df['Quantity'].sum()),
    'Avg Customer Rating'   : df['Rating'].mean().round(2),
    'Unique Products'       : df['Product'].nunique(),
    'Cities'                : df['City'].nunique(),
    'Categories'            : df['Category'].nunique(),
}

print('\n' + '='*45)
print(f'   SUPERMARKET KPIs  ({df["Date"].min().date()} → {df["Date"].max().date()})')
print('='*45)
for k, v in kpis.items():
    print(f'  {k:<28} {v:>12,.2f}' if isinstance(v, float) else f'  {k:<28} {v:>12,}')
print('='*45)

In [ ]:
# ── 4.2 Sales by Category ─────────────────────────
cat_df = (
    df.groupby('Category')
    .agg(
        Total_Sales   = ('Sales',      'sum'),
        Transactions  = ('Invoice ID', 'count'),
        Avg_Order     = ('Sales',      'mean'),
        Avg_Rating    = ('Rating',     'mean'),
        Units_Sold    = ('Quantity',   'sum'),
    )
    .round(2)
    .sort_values('Total_Sales', ascending=False)
    .reset_index()
)
cat_df['Sales_Share_%'] = (cat_df['Total_Sales'] / cat_df['Total_Sales'].sum() * 100).round(2)

print('Sales by Category:')
cat_df

In [ ]:
# ── 4.3 Sales by Branch / City ────────────────────
branch_df = (
    df.groupby(['Branch', 'City'])
    .agg(
        Total_Sales  = ('Sales',      'sum'),
        Transactions = ('Invoice ID', 'count'),
        Avg_Order    = ('Sales',      'mean'),
        Avg_Rating   = ('Rating',     'mean'),
    )
    .round(2)
    .sort_values('Total_Sales', ascending=False)
    .reset_index()
)

print('Sales by Branch & City:')
branch_df

In [ ]:
# ── 4.4 Sales by Customer Type ────────────────────
cust_df = (
    df.groupby('Customer Type')
    .agg(Total_Sales=('Sales','sum'), Transactions=('Invoice ID','count'), Avg_Order=('Sales','mean'))
    .round(2).reset_index()
)
cust_df['Sales_Share_%'] = (cust_df['Total_Sales'] / cust_df['Total_Sales'].sum() * 100).round(2)

# ── 4.5 Sales by Gender ───────────────────────────
gender_df = (
    df.groupby('Gender')
    .agg(Total_Sales=('Sales','sum'), Transactions=('Invoice ID','count'), Avg_Order=('Sales','mean'))
    .round(2).reset_index()
)
gender_df['Sales_Share_%'] = (gender_df['Total_Sales'] / gender_df['Total_Sales'].sum() * 100).round(2)

print('Customer Type:')
display(cust_df)
print('\nGender:')
display(gender_df)

In [ ]:
# ── 4.6 Sales by Payment Method ───────────────────
pay_df = (
    df.groupby('Payment')
    .agg(Total_Sales=('Sales','sum'), Transactions=('Invoice ID','count'), Avg_Order=('Sales','mean'))
    .round(2).sort_values('Total_Sales', ascending=False).reset_index()
)
pay_df['Sales_Share_%'] = (pay_df['Total_Sales'] / pay_df['Total_Sales'].sum() * 100).round(2)

print('Sales by Payment Method:')
pay_df

In [ ]:
# ── 4.7 Monthly Sales Trend ───────────────────────
monthly_df = (
    df.groupby('Month_Label')
    .agg(Total_Sales=('Sales','sum'), Transactions=('Invoice ID','count'), Avg_Order=('Sales','mean'))
    .round(2).reset_index().sort_values('Month_Label')
)

print('Monthly Sales Trend:')
monthly_df

In [ ]:
# ── 4.8 Top 10 Products ───────────────────────────
top10_df = (
    df.groupby(['Product', 'Category'])
    .agg(Total_Sales=('Sales','sum'), Transactions=('Invoice ID','count'),
         Units_Sold=('Quantity','sum'), Avg_Rating=('Rating','mean'))
    .round(2).sort_values('Total_Sales', ascending=False).head(10).reset_index()
)

print('Top 10 Products by Revenue:')
top10_df

In [ ]:
# ── 4.9 Rating Distribution ───────────────────────
bins   = [1, 2, 3, 4, 5, 5.1]
labels = ['1–2', '2–3', '3–4', '4–5', '5']
df['Rating_Band'] = pd.cut(df['Rating'], bins=bins, labels=labels, right=False, include_lowest=True)
rating_dist = df.groupby('Rating_Band', observed=True).size().reset_index(name='Count')

print('Rating Distribution:')
rating_dist

In [ ]:
# ── 4.10 Day-of-Week Pattern ──────────────────────
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_df = (
    df.groupby('Day_of_Week')
    .agg(Total_Sales=('Sales','sum'), Transactions=('Invoice ID','count'))
    .round(2).reindex(dow_order).reset_index()
)

print('Sales by Day of Week:')
dow_df

---
## 📈 Step 5 — Charts & Visualisations

In [ ]:
# ── Chart 1 · Revenue by Category (horizontal bar) ─
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    cat_df.sort_values('Total_Sales')['Category'],
    cat_df.sort_values('Total_Sales')['Total_Sales'],
    color=sns.color_palette('Blues_r', len(cat_df)),
)
for bar in bars:
    ax.text(bar.get_width() + 300, bar.get_y() + bar.get_height()/2,
            f'₹{bar.get_width():,.0f}', va='center', fontsize=9)
ax.set_xlabel('Revenue (₹)')
ax.set_title('Total Revenue by Category', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2 · Revenue Share Pie ────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    cat_df['Total_Sales'],
    labels=cat_df['Category'],
    autopct='%1.1f%%',
    colors=sns.color_palette('Set3', len(cat_df)),
    startangle=140,
    pctdistance=0.78,
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title('Revenue Share by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3 · Branch Revenue Comparison ───────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Revenue bar
palette = sns.color_palette('Blues_r', len(branch_df))
bars = axes[0].bar(branch_df['City'], branch_df['Total_Sales'], color=palette, edgecolor='white')
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 500,
                 f'₹{b.get_height():,.0f}', ha='center', fontsize=9)
axes[0].set_title('Total Revenue by City', fontweight='bold')
axes[0].set_ylabel('Revenue (₹)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# Rating bar
axes[1].bar(branch_df['City'], branch_df['Avg_Rating'],
            color=sns.color_palette('YlGn', len(branch_df)), edgecolor='white')
for i, v in enumerate(branch_df['Avg_Rating']):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
axes[1].set_title('Avg Customer Rating by City', fontweight='bold')
axes[1].set_ylabel('Avg Rating (0–5)')
axes[1].set_ylim(3.5, 4.5)

plt.suptitle('Branch & City Performance', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 4 · Customer Segmentation ───────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].pie(cust_df['Total_Sales'], labels=cust_df['Customer Type'],
            autopct='%1.1f%%', colors=['#3B82F6','#F59E0B'],
            startangle=90, wedgeprops=dict(width=0.55))
axes[0].set_title('Revenue: Member vs Normal', fontweight='bold')

axes[1].pie(gender_df['Total_Sales'], labels=gender_df['Gender'],
            autopct='%1.1f%%', colors=['#8B5CF6','#EC4899'],
            startangle=90, wedgeprops=dict(width=0.55))
axes[1].set_title('Revenue: Gender Split', fontweight='bold')

plt.suptitle('Customer Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 5 · Payment Method Bar ──────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(pay_df['Payment'], pay_df['Total_Sales'],
              color=['#3B82F6','#10B981','#8B5CF6','#F59E0B'], edgecolor='white', width=0.55)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 400,
            f'₹{b.get_height():,.0f}', ha='center', fontsize=9)
ax.set_title('Revenue by Payment Method', fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 6 · Monthly Revenue Trend (dual-axis) ───
fig, ax1 = plt.subplots(figsize=(11, 5))

ax2 = ax1.twinx()
ax1.bar(monthly_df['Month_Label'], monthly_df['Total_Sales'],
        color=ACCENT, alpha=0.4, label='Revenue')
ax2.plot(monthly_df['Month_Label'], monthly_df['Transactions'],
         color=BLUE, marker='o', linewidth=2, label='Transactions')

ax1.set_ylabel('Revenue (₹)', color=ACCENT)
ax2.set_ylabel('Transactions', color=BLUE)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax1.set_title('Monthly Revenue & Transaction Trend', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=30)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 7 · Top 10 Products ─────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    top10_df.sort_values('Total_Sales')['Product'],
    top10_df.sort_values('Total_Sales')['Total_Sales'],
    color=sns.color_palette('viridis', 10),
)
for b in bars:
    ax.text(b.get_width() + 100, b.get_y() + b.get_height()/2,
            f'₹{b.get_width():,.0f}', va='center', fontsize=9)
ax.set_xlabel('Revenue (₹)')
ax.set_title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 8 · Rating Distribution ─────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(rating_dist['Rating_Band'].astype(str), rating_dist['Count'],
              color=sns.color_palette('YlGn', len(rating_dist)), edgecolor='white')
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
            str(int(b.get_height())), ha='center', fontsize=10)
ax.set_xlabel('Rating Band')
ax.set_ylabel('Number of Transactions')
ax.set_title('Customer Rating Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 9 · Category × City Heatmap ────────────
heatmap_pivot = (
    df.groupby(['Category', 'City'])['Sales']
    .sum().round(2).unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    heatmap_pivot,
    annot=True, fmt='.0f', cmap='Blues',
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 9},
    ax=ax
)
ax.set_title('Revenue Heatmap: Category × City (₹)', fontsize=13, fontweight='bold')
ax.set_ylabel('Category')
ax.set_xlabel('City')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 10 · Day-of-Week Sales ──────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dow_df['Day_of_Week'], dow_df['Total_Sales'],
              color=sns.color_palette('Purples_r', 7), edgecolor='white')
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 100,
            f'₹{b.get_height():,.0f}', ha='center', fontsize=9, rotation=0)
ax.set_title('Revenue by Day of Week', fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue (₹)')
ax.tick_params(axis='x', rotation=20)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 11 · Category Avg Order vs Avg Rating ───
fig, ax1 = plt.subplots(figsize=(11, 5))
x = range(len(cat_df))
ax2 = ax1.twinx()

ax1.bar(x, cat_df['Avg_Order'], color=ACCENT, alpha=0.6, label='Avg Order (₹)', width=0.4)
ax2.plot(x, cat_df['Avg_Rating'], color='#F59E0B', marker='D',
         linewidth=2, markersize=8, label='Avg Rating')

ax1.set_xticks(list(x))
ax1.set_xticklabels(cat_df['Category'], rotation=25, ha='right')
ax1.set_ylabel('Avg Order Value (₹)', color=ACCENT)
ax2.set_ylabel('Avg Rating', color='#F59E0B')
ax2.set_ylim(3.5, 4.5)
ax1.set_title('Avg Order Value vs Avg Rating by Category', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

---
## 💡 Step 6 — Business Decisions & Recommendations

In [ ]:
print('\n' + '='*65)
print('       BUSINESS INSIGHTS & RECOMMENDATIONS')
print('='*65)

top_cat  = cat_df.iloc[0]
low_cat  = cat_df.iloc[-1]
top_br   = branch_df.iloc[0]
top_pay  = pay_df.iloc[0]
best_mo  = monthly_df.loc[monthly_df['Total_Sales'].idxmax(), 'Month_Label']
worst_mo = monthly_df.loc[monthly_df['Total_Sales'].idxmin(), 'Month_Label']
m_row    = cust_df[cust_df['Customer Type']=='Member'].iloc[0]
n_row    = cust_df[cust_df['Customer Type']=='Normal'].iloc[0]
avg_rtg  = df['Rating'].mean().round(2)

insights = [
    ('🏆 Best Category',
     f"{top_cat['Category']} generates ₹{top_cat['Total_Sales']:,.2f} ({top_cat['Sales_Share_%']}% of total). "
     f"Avg order ₹{top_cat['Avg_Order']:,.2f}.",
     'Increase stock depth, run loyalty discounts, expand SKU variety.'),

    ('⚠️  Under-Performing Category',
     f"{low_cat['Category']} contributes only ₹{low_cat['Total_Sales']:,.2f} ({low_cat['Sales_Share_%']}%) "
     f"despite avg rating {low_cat['Avg_Rating']}.",
     'Bundle with top categories, increase shelf visibility, run try-it promos.'),

    ('📍 Top Branch',
     f"Branch {top_br['Branch']} ({top_br['City']}) leads with ₹{top_br['Total_Sales']:,.2f} "
     f"revenue, {top_br['Transactions']} transactions, rating {top_br['Avg_Rating']}.",
     "Standardise Mumbai's operational practices chain-wide, especially for Jaipur."),

    ('💳 Top Payment Method',
     f"{top_pay['Payment']} leads with ₹{top_pay['Total_Sales']:,.2f} ({top_pay['Sales_Share_%']}%) "
     f"and highest avg order ₹{top_pay['Avg_Order']:,.2f}. Digital = 77.9% of revenue.",
     'Partner with UPI provider for cashback; ensure zero-downtime payment infra.'),

    ('👥 Member vs Normal',
     f"Members drive volume (59% txns) but Normal customers spend ₹{abs(m_row['Avg_Order']-n_row['Avg_Order']):.2f} "
     f"{'less' if m_row['Avg_Order'] < n_row['Avg_Order'] else 'more'} per order.",
     'Introduce tiered membership rewards to incentivise larger basket sizes.'),

    ('📈 Seasonal Pattern',
     f"Peak month: {best_mo} (₹{monthly_df.loc[monthly_df['Month_Label']==best_mo,'Total_Sales'].values[0]:,.2f}). "
     f"Weakest: {worst_mo}.",
     'Pre-stock Beverages, Dairy & Personal Care 3 weeks before April.'),

    ('⭐ Customer Satisfaction',
     f"Overall avg rating {avg_rtg}/5.0 — just below 4.0 benchmark. "
     f"Jaipur (3.84) and Fruits category (3.83) are lowest.",
     'Launch post-purchase feedback forms; staff training focus on Jaipur.'),
]

for title, detail, action in insights:
    print(f'\n{title}')
    print(f'   {detail}')
    print(f'   → Action: {action}')

print('\n' + '='*65)

---
## 📋 Summary Statistics

In [ ]:
# Full numeric summary
df[['Quantity', 'Unit Price', 'Sales', 'Rating']].describe().round(2)

In [ ]:
# Revenue × Gender × Customer Type cross-table
cross = (
    df.groupby(['Gender', 'Customer Type'])['Sales']
    .agg(['sum','count','mean'])
    .round(2)
    .rename(columns={'sum':'Total Revenue (₹)','count':'Transactions','mean':'Avg Order (₹)'})
)
cross

In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[['Quantity', 'Unit Price', 'Sales', 'Rating']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
> **Project:** Supermarket Sales Analytics  
> **Stack:** Python · Pandas · Matplotlib · Seaborn · Plotly · Streamlit  
> **Dashboard:** Run `streamlit run app.py` inside the `supermarket_analysis/` folder